# Day 5 — Inputs, Outputs, and Passing Data Between Tasks

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/kestra-certified/notebooks/day-05-inputs-outputs.ipynb#scrollTo=aa11bb22)

**Course:** Kestra for Data Engineers  
**Badge:** Practice  

Today you'll master the full data-passing model in Kestra: typed inputs that parameterise a flow at execution time, task output variables that chain tasks together, and internal storage for large file transfers.

**By the end of this notebook you will:**
- Define typed `inputs` (STRING, INT, BOOLEAN, DATE) with defaults and validations
- Reference inputs and task outputs using Pebble template expressions
- Use `Kestra.outputs()` to publish named values from Python tasks
- Build a three-task pipeline: generate CSV → validate → log stats
- Understand how Kestra's internal storage handles file passing

In [ ]:
%pip install -q pyyaml pandas

## 1. Typed Inputs — Parameterising a Flow

Inputs are declared at the flow level under the `inputs:` key. Each input has:
- `id` — the name used in Pebble: `{{ inputs.my_param }}`
- `type` — STRING, INT, FLOAT, BOOLEAN, DATE, DATETIME, FILE, JSON, ARRAY, DURATION
- `defaults` — optional default value (makes the input optional)
- `description` — shown in the Kestra UI

Inputs are validated before execution starts. A missing required input (no `defaults`) blocks the run immediately.

In [ ]:
import yaml

typed_inputs_flow = {
    "id": "typed-inputs-demo",
    "namespace": "tutorial.day05",
    "description": "Demonstrates all common input types with defaults",
    "inputs": [
        {
            "id": "environment",
            "type": "STRING",
            "defaults": "dev",
            "description": "Target environment: dev, staging, or prod"
        },
        {
            "id": "row_limit",
            "type": "INT",
            "defaults": 1000,
            "description": "Maximum rows to process"
        },
        {
            "id": "dry_run",
            "type": "BOOLEAN",
            "defaults": False,
            "description": "If true, run validation only without writing output"
        },
        {
            "id": "report_date",
            "type": "DATE",
            "defaults": "{{ now() | dateAdd(-1, 'DAYS') | date('yyyy-MM-dd') }}",
            "description": "Date to process (default: yesterday)"
        }
    ],
    "tasks": [
        {
            "id": "log_params",
            "type": "io.kestra.plugin.core.log.Log",
            "message": (
                "env={{ inputs.environment }} | "
                "limit={{ inputs.row_limit }} | "
                "dry_run={{ inputs.dry_run }} | "
                "date={{ inputs.report_date }}"
            )
        }
    ]
}

print(yaml.dump(typed_inputs_flow, default_flow_style=False, sort_keys=False))

> **Type safety:** If you pass `row_limit: "abc"` to an INT input, Kestra rejects the execution before any task runs. This is the first line of defence against bad pipeline arguments.

## 2. Referencing Inputs in Task Commands

Inputs are available everywhere via Pebble: in `message`, `script`, `commands`, `uri`, `headers`, and more.

Common patterns:
- Plain substitution: `{{ inputs.environment }}`
- Upper-case: `{{ inputs.environment | upper }}`
- Conditional: `{{ inputs.dry_run ? 'SKIP' : 'RUN' }}`
- Date format: `{{ inputs.report_date | date('yyyyMMdd') }}`

In [ ]:
# Simulate Pebble template rendering in Python
inputs = {
    "environment": "staging",
    "row_limit": 500,
    "dry_run": False,
    "report_date": "2026-06-09"
}

# Pebble expressions evaluated at runtime — simulated here with Python f-strings
rendered_command = (
    f"python process.py "
    f"--env {inputs['environment'].upper()} "
    f"--limit {inputs['row_limit']} "
    f"--date {inputs['report_date'].replace('-', '')} "
    f"{'--dry-run' if inputs['dry_run'] else '--write'}"
)

print("Rendered command:")
print(rendered_command)

print("\nKestra YAML equivalent:")
cmd_example = {
    "id": "process_data",
    "type": "io.kestra.plugin.scripts.shell.Commands",
    "commands": [
        "python process.py "
        "--env {{ inputs.environment | upper }} "
        "--limit {{ inputs.row_limit }} "
        "--date {{ inputs.report_date | date('yyyyMMdd') }} "
        "{{ inputs.dry_run ? '--dry-run' : '--write' }}"
    ]
}
print(yaml.dump(cmd_example, default_flow_style=False, sort_keys=False))

## 3. Task Outputs — The Output Variable System

Each Kestra task type publishes specific output variables. Reference them downstream with:

```yaml
{{ outputs.<taskId>.<outputKey> }}
```

| Task type | Output key | Value |
|-----------|-----------|-------|
| `core.http.Request` | `.body` | Response body string |
| `core.http.Request` | `.code` | HTTP status int |
| `scripts.python.Commands` | `.vars.key` | Value from `Kestra.outputs({key: val})` |
| `scripts.shell.Commands` | `.outputFiles['name']` | Internal storage URI |
| `core.http.Download` | `.uri` | Internal storage URI of downloaded file |

In [ ]:
# Illustrate task output chaining
output_chain_flow = {
    "id": "output-chain-demo",
    "namespace": "tutorial.day05",
    "tasks": [
        {
            "id": "fetch_data",
            "type": "io.kestra.plugin.core.http.Request",
            "uri": "https://jsonplaceholder.typicode.com/todos?userId=1&_limit=5",
            "method": "GET"
        },
        {
            "id": "analyse",
            "type": "io.kestra.plugin.scripts.python.Commands",
            "beforeCommands": [],
            "env": {"BODY": "{{ outputs.fetch_data.body }}"},
            "script": (
                "import json, os\n"
                "todos = json.loads(os.environ['BODY'])\n"
                "completed = sum(1 for t in todos if t['completed'])\n"
                "Kestra.outputs({'total': len(todos), 'completed': completed, 'pending': len(todos) - completed})\n"
            )
        },
        {
            "id": "log_result",
            "type": "io.kestra.plugin.core.log.Log",
            "message": (
                "Total: {{ outputs.analyse.vars.total }} | "
                "Done: {{ outputs.analyse.vars.completed }} | "
                "Pending: {{ outputs.analyse.vars.pending }}"
            )
        }
    ]
}

print(yaml.dump(output_chain_flow, default_flow_style=False, sort_keys=False))

## 4. Kestra.outputs() — Publishing from Python

`Kestra.outputs(dict)` is the bridge from Python to Kestra's variable system. The Kestra worker injects this function into every Python task environment — no import needed.

In [ ]:
import pandas as pd
import json

# Simulate what runs inside a Kestra Python task
url = "https://people.sc.fsu.edu/~jburkardt/data/csv/airtravel.csv"
df = pd.read_csv(url)

# Clean column names
df.columns = [c.strip() for c in df.columns]
numeric_cols = df.select_dtypes(include='number').columns.tolist()

stats = {
    "row_count": len(df),
    "column_count": len(df.columns),
    "columns": list(df.columns),
    "numeric_columns": numeric_cols
}

# In Kestra, this line publishes outputs.analyse.vars.*
# Kestra.outputs(stats)  ← injected by runtime
print("Kestra.outputs() payload:")
print(json.dumps(stats, indent=2))

print("\nDownstream Pebble references:")
print("  {{ outputs.analyse.vars.row_count }}      →", stats['row_count'])
print("  {{ outputs.analyse.vars.column_count }}   →", stats['column_count'])

## 5. Internal Storage — Passing Files Between Tasks

When tasks need to exchange files (CSV, Parquet, images), Kestra's internal storage handles it transparently. Files are stored in an S3-compatible object store and referenced by opaque URI.

**Write a file:** Use `outputFiles: ['*.csv']` in a Shell or Python task  
**Read a file:** Reference `{{ outputs.taskId.outputFiles['file.csv'] }}` as an `inputFiles` URI in the next task

In [ ]:
# Three-task pipeline: generate → validate → report
file_pipeline_flow = {
    "id": "file-pipeline",
    "namespace": "tutorial.day05",
    "description": "Task A generates CSV, Task B validates it, Task C reports stats",
    "inputs": [
        {"id": "row_count", "type": "INT", "defaults": 100}
    ],
    "tasks": [
        {
            "id": "generate_csv",
            "type": "io.kestra.plugin.scripts.python.Commands",
            "beforeCommands": ["pip install pandas -q"],
            "script": (
                "import pandas as pd, os\n"
                "n = int(os.environ.get('ROW_COUNT', 100))\n"
                "df = pd.DataFrame({'id': range(n), 'value': [i * 1.5 for i in range(n)]})\n"
                "df.to_csv('output.csv', index=False)\n"
                "print(f'Generated {n} rows')\n"
            ),
            "env": {"ROW_COUNT": "{{ inputs.row_count }}"},
            "outputFiles": ["output.csv"]
        },
        {
            "id": "validate_csv",
            "type": "io.kestra.plugin.scripts.python.Commands",
            "beforeCommands": ["pip install pandas -q"],
            "inputFiles": {"data.csv": "{{ outputs.generate_csv.outputFiles['output.csv'] }}"},
            "script": (
                "import pandas as pd\n"
                "df = pd.read_csv('data.csv')\n"
                "assert 'id' in df.columns, 'Missing id column'\n"
                "assert 'value' in df.columns, 'Missing value column'\n"
                "assert df['value'].notna().all(), 'Null values found'\n"
                "Kestra.outputs({'rows': len(df), 'valid': True})\n"
                "print(f'Validation passed: {len(df)} rows, 0 nulls')\n"
            )
        },
        {
            "id": "report_stats",
            "type": "io.kestra.plugin.core.log.Log",
            "message": "Pipeline complete — {{ outputs.validate_csv.vars.rows }} rows validated: {{ outputs.validate_csv.vars.valid }}"
        }
    ]
}

print(yaml.dump(file_pipeline_flow, default_flow_style=False, sort_keys=False))

> `inputFiles` maps a local filename to an internal storage URI from a previous task. Kestra downloads the file into the task's working directory before execution begins — the Python script just opens `'data.csv'` as a normal file.

## 6. Simulate the Full Pipeline Locally

In [ ]:
import pandas as pd, io

row_count = 50  # inputs.row_count

# Task A: generate_csv
df_gen = pd.DataFrame({'id': range(row_count), 'value': [i * 1.5 for i in range(row_count)]})
csv_buffer = df_gen.to_csv(index=False)
print(f"Task A — generated {row_count} rows")
print(df_gen.head(3).to_string())

# Task B: validate_csv  (inputFiles injects the CSV)
df_val = pd.read_csv(io.StringIO(csv_buffer))
assert 'id' in df_val.columns
assert 'value' in df_val.columns
assert df_val['value'].notna().all()
validate_outputs = {'rows': len(df_val), 'valid': True}
print(f"\nTask B — validation passed: {validate_outputs}")

# Task C: report_stats
log_msg = f"Pipeline complete — {validate_outputs['rows']} rows validated: {validate_outputs['valid']}"
print(f"\nTask C — Log: {log_msg}")

## 7. JSON and ARRAY Inputs

In [ ]:
# Advanced input types: JSON and ARRAY
advanced_inputs_flow = {
    "id": "advanced-inputs",
    "namespace": "tutorial.day05",
    "inputs": [
        {
            "id": "config",
            "type": "JSON",
            "defaults": {"threshold": 0.05, "method": "pearson"},
            "description": "Analysis configuration object"
        },
        {
            "id": "columns_to_drop",
            "type": "ARRAY",
            "itemType": "STRING",
            "defaults": ["created_at", "updated_at"],
            "description": "Columns to exclude from analysis"
        }
    ],
    "tasks": [
        {
            "id": "log_config",
            "type": "io.kestra.plugin.core.log.Log",
            "message": (
                "threshold={{ inputs.config.threshold }} | "
                "method={{ inputs.config.method }} | "
                "drop={{ inputs.columns_to_drop }}"
            )
        }
    ]
}

print(yaml.dump(advanced_inputs_flow, default_flow_style=False, sort_keys=False))

## 8. Putting It All Together — Parameterised ETL Flow

In [ ]:
# Complete parameterised ETL flow demonstrating inputs + outputs + file passing
etl_flow = {
    "id": "parameterised-etl",
    "namespace": "tutorial.day05",
    "description": "Configurable ETL: download → transform → validate → report",
    "inputs": [
        {"id": "source_url", "type": "STRING",
         "defaults": "https://people.sc.fsu.edu/~jburkardt/data/csv/airtravel.csv"},
        {"id": "min_rows", "type": "INT", "defaults": 10},
        {"id": "environment", "type": "STRING", "defaults": "dev"}
    ],
    "tasks": [
        {
            "id": "download",
            "type": "io.kestra.plugin.core.http.Download",
            "uri": "{{ inputs.source_url }}"
        },
        {
            "id": "transform",
            "type": "io.kestra.plugin.scripts.python.Commands",
            "beforeCommands": ["pip install pandas -q"],
            "inputFiles": {"raw.csv": "{{ outputs.download.uri }}"},
            "script": (
                "import pandas as pd\n"
                "df = pd.read_csv('raw.csv')\n"
                "df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]\n"
                "df.to_csv('transformed.csv', index=False)\n"
                "Kestra.outputs({'rows': len(df), 'columns': list(df.columns)})\n"
            ),
            "outputFiles": ["transformed.csv"]
        },
        {
            "id": "validate",
            "type": "io.kestra.plugin.scripts.python.Commands",
            "beforeCommands": ["pip install pandas -q"],
            "inputFiles": {"data.csv": "{{ outputs.transform.outputFiles['transformed.csv'] }}"},
            "env": {"MIN_ROWS": "{{ inputs.min_rows }}"},
            "script": (
                "import pandas as pd, os\n"
                "df = pd.read_csv('data.csv')\n"
                "min_rows = int(os.environ['MIN_ROWS'])\n"
                "assert len(df) >= min_rows, f'Only {len(df)} rows, need {min_rows}'\n"
                "null_pct = df.isnull().mean().max()\n"
                "Kestra.outputs({'valid': True, 'null_pct': round(float(null_pct), 4)})\n"
            )
        },
        {
            "id": "report",
            "type": "io.kestra.plugin.core.log.Log",
            "message": (
                "[{{ inputs.environment | upper }}] ETL complete — "
                "{{ outputs.transform.vars.rows }} rows, "
                "max null%={{ outputs.validate.vars.null_pct }}, "
                "valid={{ outputs.validate.vars.valid }}"
            )
        }
    ]
}

print(yaml.dump(etl_flow, default_flow_style=False, sort_keys=False))

## Challenge

Build a Kestra flow that:
1. Takes two inputs: `dataset_url` (STRING, required) and `max_null_pct` (FLOAT, default `0.1`)
2. Downloads the CSV using `io.kestra.plugin.core.http.Download`
3. Runs a Python task that computes per-column null percentage and calls `Kestra.outputs({'summary': {...}})` with a dict of column → null_pct
4. Logs: `"Columns with nulls above threshold: {{ outputs.check.vars.summary }}"`

Test with: `dataset_url = 'https://people.sc.fsu.edu/~jburkardt/data/csv/airtravel.csv'`

In [ ]:
# Your challenge solution here
challenge_flow = {
    "id": "null-checker",
    "namespace": "tutorial.day05.challenge",
    "inputs": [
        # Add your inputs here
    ],
    "tasks": [
        # Task 1: download
        # Task 2: check nulls
        # Task 3: log
    ]
}
print(yaml.dump(challenge_flow, default_flow_style=False, sort_keys=False))

## Recap

| Concept | Kestra YAML | Pebble reference |
|---------|-------------|------------------|
| STRING input | `type: STRING` | `{{ inputs.name }}` |
| INT input with default | `type: INT, defaults: 100` | `{{ inputs.limit }}` |
| Python output | `Kestra.outputs({'k': v})` | `{{ outputs.id.vars.k }}` |
| HTTP response body | `type: core.http.Request` | `{{ outputs.id.body }}` |
| File from previous task | `outputFiles: ['*.csv']` | `{{ outputs.id.outputFiles['f.csv'] }}` |
| Inject file into task | `inputFiles: {local.csv: URI}` | (available as local file) |

**Tip:** Kestra's internal storage is an S3-compatible object store. Large outputs (files, DataFrames) are automatically stored there and referenced by URI — you never need to worry about serialization.

**Tomorrow — Day 6:** Error Handling — retries, timeouts, flow-level error handlers, and `allowFailure`.